In [1]:
from dotenv import load_dotenv
import os 
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'true' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'RAG'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])

43b13g4OZS
gpt-5-mini
랭스미스로 추적 중입니다 : lsv2_pt_24


### 간단한 Human in the Loop

Human in the Loop는 랭그래프의 흐름 속에 사람과 소통하는 포인트를 하나 추가하는 것입니다.


In [65]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage, AIMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    y_or_n: str
    log: Annotated[list, add_messages]

def hello(state: State) -> State:
    print('안녕하세요? 제가 마음에 드시나요?')
    return {"log": '[hello]'}

def answer(state: State) -> State:
    ans = input('마음에 들면 y, 아니면 n (y/n) ')

    return {"y_or_n": ans.strip().lower(), "log": f'[hello : {ans}]'}
    
def sad(state: State) -> State:

    print('저한테 다시 기회를 주세요...')
    return {"log": '[sad]'}

def happy(state: State) -> State:
    print('감사합니다! 좋은 하루 되세요!')
    return {"log": '[happy]'}

In [66]:
g = StateGraph(State)

g.add_node("hello", hello)
g.add_node("answer", answer)
g.add_node("sad", sad)
g.add_node("happy", happy)

g.add_edge(START, "hello")
g.add_edge("hello", "answer")

def route(state: State) -> str:
    return "happy" if state.get("y_or_n", "") == "y" else "sad"

g.add_conditional_edges("answer", route, {"happy": "happy", "sad": "sad"})
g.add_edge("happy", END)
g.add_edge("sad", END)

app = g.compile()

In [ ]:
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [67]:
final = app.invoke({"log": []})
print(final["log"])

안녕하세요? 제가 마음에 드시나요?
저한테 다시 기회를 주세요...
[HumanMessage(content='[hello]', additional_kwargs={}, response_metadata={}, id='168258e4-81da-49aa-88cb-f2a641dbcb37'), HumanMessage(content='[hello : ]', additional_kwargs={}, response_metadata={}, id='457291f8-a5a8-4ef5-aa5e-1afa9e8cacbf'), HumanMessage(content='[sad]', additional_kwargs={}, response_metadata={}, id='27fe0171-95bf-4618-a106-e4a716e9df79')]


## Adaptive RAG

llm이 입력에 따라 적절한 도구를 선택할 수 있게하는 adaptive 방법을 연습해봅시다.

In [24]:
import os, glob
from typing import TypedDict, Literal, List, Annotated

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

### 1. State

해당 상태에는 

- 대화의 맥락을 이해할 수 있게하는 message
- 두 분기 중 하나를 선택하게 하는 route
- RAG를 위한 context와 refs

In [25]:
class GState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], add_messages]   # 대화 이력 누적
    route: Literal["common","rag"]                         # 라우팅 결과
    context: str                                           # RAG 컨텍스트
    refs: List[str]                                        # RAG 출처 ["file.pdf p.N", ...]

### 2. 데이터 로더 및 저장, 리트리버 설정

In [26]:
from langchain_openai import AzureOpenAIEmbeddings
emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,                      
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)

def build_vectordb(pdf_glob: str, chunk_size=800, chunk_overlap=120) -> InMemoryVectorStore:
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = []
    for path in glob.glob(pdf_glob):
        loader = PyMuPDFLoader(path)   
        docs.extend(loader.load())

    splits = splitter.split_documents(docs) 
    return InMemoryVectorStore.from_documents(splits, embedding=emb)

VSTORE = build_vectordb("pdf/*.pdf")
retriever = VSTORE.as_retriever(search_kwargs={"k": 10}) 

### 3. 다양한 용도의 LLM


In [27]:
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.1,
)

### 4. Message에서 최신 질문을 추출하는 함수

message에는 human과 ai가 반복적으로 저장이됩니다. 

가장 마지막 index의 message가 최신 기록입니다. 

In [28]:
def last_user_text(messages: List[BaseMessage]) -> str:
    for m in reversed(messages):
        if isinstance(m, HumanMessage):
            return m.content
    return messages[-1].content if messages else ""

### 5. Router + Human Check

일반적인 챗봇으로 동작할지 아니면 문서를 기반으로 정확한 정보를 줘야하는지 지능적으로 판단

In [29]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 사용자의 질문이 이용자 약관을 검토해야 하는 '엄밀한 근거(약관/조항/정의/의무/해지/요금/계약 등)'를 필요로 하면 rag, "
     "그 외 통신 일반 상식/조언이면 common을 출력한다. 출력은 반드시 'rag' 또는 'common' 한 단어만 말해야한다."),
    ("human", "질의: {q}")
])

def route_node(state: GState) -> GState:
    """RAG/일반 대화 모드 라우팅"""
    q = last_user_text(state["messages"])
    out = (router_prompt | llm | StrOutputParser()).invoke({"q": q}).strip().lower()
    mode = "rag" if out.startswith("rag") else "common"  
    return {"route": mode}

def human_check_node(state: GState) -> GState:
    """사용자가 rag 사용을 승인하는 노드"""
    u_message = last_user_text(state["messages"])
    print(u_message)
    approve = input("RAG 모드로 진행할까요? (y/n): ").strip().lower()
    if approve == "y":
        print("RAG 모드로 진행합니다.")
        return {"route": "rag"}
    else:
        print("RAG 모드를 취소합니다.")
        return {"route": "common"}


### 6. 리트리버

검색과 함께 근거를 제시할 ref도 metadata에서 추출한다.

In [12]:
def retrieve_node(state: GState) -> GState:
    """RAG 컨텍스트 및 레퍼런스 추출"""
    q = last_user_text(state["messages"])
    docs = retriever.invoke(q)                        

    # 컨텍스트 텍스트
    context = "\n\n---\n\n".join(d.page_content for d in docs)

    # 레퍼런스 추출 (파일명 + 페이지)
    refs: List[str] = []
    
    seen = set()
    for d in docs:
        meta = d.metadata or {}
        src = meta.get("source") or meta.get("file_path") or meta.get("path")
        page = meta.get("page")
        if page is None:
            page = meta.get("page_number")
        disp_page = (page + 1) if isinstance(page, int) and page >= 0 else None
        if src:
            fname = os.path.basename(src)
            ref = f"{fname} p.{disp_page}" if disp_page is not None else f"{fname}"
            key = (fname, disp_page)
            if key not in seen:
                seen.add(key)
                refs.append(ref)

    return {"context": context, "refs": refs}

### 7. 이용 약관 상담 RAG 

In [30]:
no_ref = "해당 질문에 대한 자료를 찾을 수 없습니다. 담당자에게 문의 바랍니다."

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사실을 근거로 대답하는 RAG이다. 아래 컨텍스트의 내용만을 근거로 답하라. "
               f"근거가 부족하면 '{no_ref}'이라고만 답하라.\n\n"
               "컨텍스트:\n{context}"),
    MessagesPlaceholder("messages"),   
])

def answer_rag(state: GState) -> GState:
    """RAG 모드 엄밀한 답변 생성"""
    out = (rag_prompt | llm | StrOutputParser()).invoke({
        "messages": state["messages"],
        "context": state.get("context","")
    }).strip()

    # 레퍼런스 붙이기
    refs = state.get("refs", [])
    if out.strip() == no_ref:
        refs = []
    if refs:
        out = out.rstrip() + "\n\n[출처] " + "; ".join(refs[:3])

    return {"messages": [AIMessage(content=out)]}

### 8. 가벼운 대화 챗봇

In [31]:
common_prompt = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하고 명확하고 친절하게, 비즈니스 톤으로 답하라. 가벼운 질문에는 편안한 어조를 허용한다."),
    MessagesPlaceholder("messages"),   
])

def answer_without_rag(state: GState) -> GState:
    """RAG 없이 일반 대화 모드"""
    out = (common_prompt | llm | StrOutputParser()).invoke({
        "messages": state["messages"]
    })
    return {"messages": [AIMessage(content=out.strip())]}

### 9. 랭그래프 구성

In [32]:
graph = StateGraph(GState)
graph.add_node("route", route_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("answer_without_rag", answer_without_rag)
graph.add_node("answer_rag", answer_rag)
# Human in the Loop 노드 추가
graph.add_node("human_check", human_check_node)

graph.set_entry_point("route")
graph.add_conditional_edges(
    "route",
    lambda s: s["route"],
    {"common": "answer_without_rag", "rag": "human_check"} # rag 하기 전에 사람이 승인
)

# Human in the Loop 엣지를 추가
graph.add_conditional_edges(
    "human_check",
    lambda s: s["route"],
    {"common": "answer_without_rag", "rag": "retrieve"}
)

graph.add_edge("retrieve", "answer_rag")
graph.add_edge("answer_without_rag", END)
graph.add_edge("answer_rag", END)

In [33]:
# multi-turn 대화 이력 누적을 위한 메시지 누적기
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [39]:
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

### 10. 실행 함수

In [40]:
def run(sid: str, text: str) -> str:

    res = app.invoke(
        {"messages": [HumanMessage(content=text)]},
        config={"configurable": {"thread_id": sid}},
    )
    return res["messages"][-1].content  

sid = "user_001"

In [41]:
print(run(sid, "내 이름은 김진태야")) 

안녕하세요, 김진태님. 만나서 반갑습니다. 어떻게 도와드릴까요?


In [42]:
print(run(sid, "내 이름이 뭐라고 했는지 기억해?")) 

네, 김진태님이라고 말씀해 주셨습니다. 도움이 필요하시면 언제든 말씀해 주세요!


In [43]:
print(run(sid, "GiGA WiFi Buddy 요금제에 대해서 설명해줘.")) 

GiGA WiFi Buddy 요금제에 대해서 설명해줘.
RAG 모드를 취소합니다.
GiGA WiFi Buddy 요금제는 KT에서 제공하는 무선 인터넷 서비스입니다. 주로 가정이나 소규모 사무실에서 빠르고 안정적인 WiFi 환경을 구축할 수 있도록 설계되었습니다. 주요 특징은 다음과 같습니다.

1. 초고속 인터넷 제공: GiGA급 속도로 빠른 인터넷 사용이 가능합니다.
2. 무선 공유기 포함: WiFi Buddy 전용 공유기가 제공되어 별도의 장비 구매 없이 바로 사용 가능.
3. 다양한 요금제 옵션: 사용량과 속도에 따라 선택할 수 있는 요금제가 마련되어 있습니다.
4. 간편한 설치 및 관리: 전문 설치 기사 방문 없이도 쉽게 설치할 수 있고, 전용 앱으로 관리가 편리합니다.

정확한 요금과 상세 조건은 KT 공식 홈페이지나 고객센터를 통해 확인하시는 것을 권장드립니다. 추가로 궁금한 점 있으시면 말씀해 주세요!


## Iterative RAG  with Human in the Loop

이번 예제 llm이 한 답변을 평가하고 만약 충분하지 못할 경우 query를 재작성해서 다시 질의하는 것을 반복하는 RAG 모델입니다.

하지만 질의를 평가하는 것을 온전히 인공지능에 맞기게 되면 오히려 이상한 쿼리를 만들어 낼 수 있습니다.

In [44]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore 

### 1. State 정의

State의 구성은 다음과 같습니다.

1. State (S)
    - origin_q: 사용자의 원질문(보존)
    - refine_q: 검색 최적화를 위한 재작성 질문 --> refine이 업데이트
    - context: 검색된 컨텍스트(문서 본문들) --> retrieve가 벡터DB로부터 추출
    - step: 반복 횟수(예산/루프 제한) --> refine될 때마다 1씩 추가
    - answer: 현재 초안 or 최종 답 --> 반복중에는 draft가 답을 작성하고 최종 마지막에 finalize가 작성
    - done: 종료 여부 플래그 --> judge가 반복 수행할지 판단

In [45]:
# State 정의
class S(TypedDict):
    origin_q: str
    refine_q: str
    context: List[str]
    step: int
    answer: str
    iter_limit: int
    done: bool

### 2. Data Split & Store & Retriver

이번 예제에서는 이해를 돕기 위해 가짜로 작성한 dummy doc을 사용하겠습니다.

만약 필요하다면 이 부분의 loader만 추가한다면 외부 문서를 활용한 RAG가 될 것입니다.

In [46]:
# 더미 문서
dummy_docs = [
    Document(page_content="주식회사 KJT는 최근 파란색 사과의 재배를 성공했다.", metadata={"source": "news"}),
    Document(page_content="파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.", metadata={"source": "wiki"}),
    Document(page_content="곳 (주)KJT는 무지개 사과를 개발할 계획이라고 김진태 연구원은 발표했다.", metadata={"source": "news"}),
    Document(page_content="보라색 사과에는 단백질이 풍부하다.", metadata={"source": "news"}),
    Document(page_content="연구팀은 붉은빛 무늬가 나타나는 검은 사과의 유전자 구조를 해독했다고 밝혔다.", metadata={"source": "news"}),
    Document(page_content="은빛 사과는 최근 패션업계에서 스타 이춘자씨가 주얼리 소재로 활용해 인기를 끌었다.", metadata={"source": "magazine"}),
    Document(page_content="청록색 사과는 바다에서 재배되는데 해양 미네랄이 풍부해 건강식품으로 각광받고 있다.", metadata={"source": "news"}),
    Document(page_content="투명 사과는 전설 속에서 금지된 열매로 묘사되지만, 최근 재현 실험이 진행 중이다.", metadata={"source": "wiki"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10, separators=["\n\n", "\n", "  ", " "])
splits = splitter.split_documents(dummy_docs)

#  벡터스토어
emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,                      
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)
vectordb = InMemoryVectorStore.from_documents(splits, embedding=emb)
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 2})

### 3. LLM 

생성 및 지능적 판단을 위해 사용할 공통적인 llm을 선언합니다.

In [48]:
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.2,
)

### 4. Retrieve

State의 쿼리에 대해 DB에서 정보를 검색하는 부분입니다. 

In [49]:
def retrieve(state: S) -> S:
    """
    질의에 대한 문서(context)를 VectorDB에서 검색
    context 업데이트
    """
    if state["step"] == 0:
        q = state["origin_q"]
    else:
        q = state["refine_q"]
    docs = retriever.invoke(q)
    state["context"] = [d.page_content for d in docs]
    return state

####  5. Draft

주어진 정보를 취합해서 초안을 생성하는 RAG 부분입니다. 

In [50]:
draft_prompt = ChatPromptTemplate.from_messages([
    ("system","아래 컨텍스트만 근거로 간결히 답하라."),
    ("human","질문: {refine_q}\n\n컨텍스트:\n{context}")
])

def draft(state: S) -> S:
    
    # 초회라면 refine_q를 origin_q로 맞춤
    if state["step"]==0:
        state["refine_q"] = state["origin_q"]

    out = (draft_prompt | llm | StrOutputParser()).invoke({
        "refine_q": state["refine_q"],
        "context": "\n\n".join(state["context"])
    })
    state["answer"] = out
    return state

### 6. Judge

초안을 검토해서 재작성 여부를 판단하는 노드입니다.

무한히 반복되는 것을 방지하기 위해 step의 크기를 보고 done을 강제로 설정합니다.

In [52]:
def judge(state: S) -> S:

    # 최대 반복 제한 n회
    if state["step"] >= state["iter_limit"]:
        state["done"] = True
        return state
    
    prompt = ChatPromptTemplate.from_template(
        "다음 답변이 질문에 대해 충분한가? 전혀 틀렸으면 no, 적당히 내용이 포함되어 있으면 yes 만 답하라. (yes/no)\n"
        "반드시 yes 또는 no로만 대답하라.\n"
        "질문: {refine_q}\n답변: {answer}"
    )
    
    chain = prompt | llm | StrOutputParser()

    j = chain.invoke({
        "refine_q": state["refine_q"],
        "answer": state["answer"]
    })
    
    state['done'] = "yes" in j
    return state


### 7. Refine

judge에 의해 결과가 좋지 못하다고 판단되면 쿼리를 수정한다. 

그리고 다시 루프가 원래대로 돌아가므로 step을 하나 올린다.

In [55]:
def refine(state: S) -> S:
    prompt = ChatPromptTemplate.from_template(
        "다음 질문을 더 잘 검색되도록 재작성하거나 필요한 경우 1개 서브질문으로 바꿔라. 문장 하나로만 작성하라.\n"
        "원질문: {refine_q}\n현재답: {answer}\n컨텍스트(일부): {context}"
    )

    chain = prompt | llm | StrOutputParser()
    r = chain.invoke({
        "refine_q": state["refine_q"],
        "answer": state["answer"],
        "context": "\n\n".join(state["context"])
    })
    state["refine_q"] = r
    state["step"] += 1
    return state

### 8. Finalize

만약 답변이 충분하거나 iter가 충분히 반복되었다면 초안을 토대로 최종적인 답만 간결하게 추출한다.

In [56]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 최종 답안을 출력한다. '묻는 말만' 답을 완성된 문장으로 간결·정확하게 답하라. "
     "사족/메타발언/면책문구 금지. 컨텍스트 밖 추론 금지. "
     "근거가 부족하면 '근거가 부족해 답할 수 없습니다.'이라고만 답하라."),
    ("human","질문: {origin_q}\n초안: {draft}\n컨텍스트:\n{context}")
])


def finalize(state: S) -> S:

    out = (final_prompt | llm | StrOutputParser()).invoke({
        "origin_q": state["origin_q"],
        "draft": state["answer"],
        "context": "\n\n".join(state["context"])
    })
    state["answer"] = out
    return state

### 9. 그래프 구성

만들어진 노드를 edge로 연결하여 lang graph를 compile한다.

entry와 end를 주의하여 작성

조건부 분기에 done을 통해 어떤 node로 갈지를 지정한다.


In [57]:
g = StateGraph(S)
g.add_node("retrieve", retrieve)
g.add_node("draft", draft)
g.add_node("judge", judge)
g.add_node("refine", refine)
g.add_node("finalize", finalize)

g.set_entry_point("retrieve")  # 시작 노드 지정
g.add_edge("retrieve","draft")
g.add_edge("draft","judge")
g.add_conditional_edges(
    "judge", # judge가 분기를 판단
    lambda s: "finalize" if s["done"] else "refine", # done=True면 finalize, False면 refine
    {"finalize": "finalize", "refine": "refine"} # 각 조건에 맞는 다음 노드 지정
)
g.add_edge("refine","retrieve")
g.add_edge("finalize", END)

app = g.compile()

### 그래프 구조 확인

In [58]:
# plot graph
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [59]:
# ---- 실행 예시
if __name__ == "__main__":
    result = app.invoke({
        "origin_q":"파란색 사과의 상징은?", 
        "refine_q": "", 
        "context":[], 
        "step":0, 
        "answer":"", 
        "iter_limit":3,
        "done":False
    })
    print(result["answer"])

파란색 사과는 발전과 혁신을 상징한다.


In [60]:
init_state = {"origin_q": "파란색 사과를 개발한 회사는 어디인가?", "refine_q": "", "context": [], "step":0, "answer":"", "iter_limit": 3, "done":False}
result = app.invoke(init_state)
print(f'iteration : {result["step"]} step')  # 몇 번 반복했는지
print(f'refined q : {result["refine_q"]}')  # 수정된 질문
print(f'context : {result["context"]}')  # 찾은 원문
print('\n--- answer ---\n')
print(result['answer'])  # 최종 답변 출력

iteration : 0 step
refined q : 파란색 사과를 개발한 회사는 어디인가?
context : ['주식회사 KJT는 최근 파란색 사과의 재배를 성공했다.', '파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.']

--- answer ---

파란색 사과를 개발한 회사는 주식회사 KJT이다.


In [61]:
init = {"origin_q":"파란색 사과의 상징은?","refine_q":"", "context":[], "step":0, "answer":"", "iter_limit": 3, "done":False}

for idx, step_update in enumerate(app.stream(init, stream_mode="updates")):
    print(f"[iter {idx}] -- {step_update}")

[iter 0] -- {'retrieve': {'origin_q': '파란색 사과의 상징은?', 'refine_q': '', 'context': ['파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.', '보라색 사과에는 단백질이 풍부하다.'], 'step': 0, 'answer': '', 'iter_limit': 3, 'done': False}}
[iter 1] -- {'draft': {'origin_q': '파란색 사과의 상징은?', 'refine_q': '파란색 사과의 상징은?', 'context': ['파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.', '보라색 사과에는 단백질이 풍부하다.'], 'step': 0, 'answer': '파란색 사과는 발전과 혁신을 상징한다.', 'iter_limit': 3, 'done': False}}
[iter 2] -- {'judge': {'origin_q': '파란색 사과의 상징은?', 'refine_q': '파란색 사과의 상징은?', 'context': ['파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.', '보라색 사과에는 단백질이 풍부하다.'], 'step': 0, 'answer': '파란색 사과는 발전과 혁신을 상징한다.', 'iter_limit': 3, 'done': True}}
[iter 3] -- {'finalize': {'origin_q': '파란색 사과의 상징은?', 'refine_q': '파란색 사과의 상징은?', 'context': ['파랑새는 희망을 상징하지만 파란색 사과는 그것에 더한 의미가 있다. 바로 발전과 혁신을 상징한다.', '보라색 사과에는 단백질이 풍부하다.'], 'step': 0, 'answer': '파란색 사과는 발전과 혁신을 상징한다.', 'iter_limit': 3, 'done': True}}


In [84]:
init = {"origin_q":"색깔 사과!","refine_q":"", "context":[], "step":0, "answer":"", "iter_limit": 3, "done":False}

for idx, step_update in enumerate(app.stream(init, stream_mode="updates")):
    print(f"[iter {idx}] -- {step_update}")

print(f'\n\n{step_update['finalize']['answer']}')

[iter 0] -- {'retrieve': {'origin_q': '색깔 사과!', 'refine_q': '', 'context': ['보라색 사과에는 단백질이 풍부하다.', '은빛 사과는 최근 패션업계에서 스타 이춘자씨가 주얼리 소재로 활용해 인기를 끌었다.'], 'step': 0, 'answer': '', 'done': False}}
[iter 1] -- {'draft': {'origin_q': '색깔 사과!', 'refine_q': '색깔 사과!', 'context': ['보라색 사과에는 단백질이 풍부하다.', '은빛 사과는 최근 패션업계에서 스타 이춘자씨가 주얼리 소재로 활용해 인기를 끌었다.'], 'step': 0, 'answer': '보라색 사과는 단백질이 풍부하고, 은빛 사과는 주얼리 소재로 인기가 있습니다.', 'iter_limit': 3, 'done': False}}
[iter 2] -- {'judge': {'origin_q': '색깔 사과!', 'refine_q': '색깔 사과!', 'context': ['보라색 사과에는 단백질이 풍부하다.', '은빛 사과는 최근 패션업계에서 스타 이춘자씨가 주얼리 소재로 활용해 인기를 끌었다.'], 'step': 0, 'answer': '보라색 사과는 단백질이 풍부하고, 은빛 사과는 주얼리 소재로 인기가 있습니다.', 'iter_limit': 3, 'done': False}}
[iter 3] -- {'refine': {'origin_q': '색깔 사과!', 'refine_q': '보라색 사과의 영양 성분과 은빛 사과가 주얼리 소재로 인기를 끄는 이유는 무엇인가요?', 'context': ['보라색 사과에는 단백질이 풍부하다.', '은빛 사과는 최근 패션업계에서 스타 이춘자씨가 주얼리 소재로 활용해 인기를 끌었다.'], 'step': 1, 'answer': '보라색 사과는 단백질이 풍부하고, 은빛 사과는 주얼리 소재로 인기가 있습니다.', 'iter_limit': 3, 'done': False}}
[iter

In [62]:
def chat(session_id: str, question: str) -> str:

    result = app.invoke(
        {
            "origin_q": question,
            "refine_q": "",
            "context": [],
            "step": 0,
            "answer": "",
            "iter_limit": 3,
            "done": False
        },
        config={"configurable": {"thread_id": session_id}},
    )
    return result["answer"]

### Practice

- 프롬프트를 변경해봅시다.
- 리트리버의 유사도 검색을 mmr로 변경해봅시다.

### Advaned
- Iterative와 Adaptive를 결합해봅시다.